In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
#calculate accuracy for each poem
df_poem = pd.read_json('500samples_RESULTS_aya23_extracted.json', lines=True)

# Define a function to calculate accuracy for each row
def calculate_accuracy(row):
    labels = row['labels']
    extracted = row['extracted_pos_tags']
    
    # Count how many POS tags match
    correct_tags = sum(1 for label, extracted_tag in zip(labels, extracted) if label == extracted_tag)
    
    # Calculate accuracy as a percentage
    accuracy = correct_tags / len(labels) *100 if len(labels) > 0 else 0
    return accuracy

# Apply the accuracy calculation for each row
df_poem['accuracy'] = df_poem.apply(calculate_accuracy, axis=1)
print(df_poem.head())

#df_poem.to_json('500_samples_RESULTS_extracted_accuracy_AYA.json', orient='records', lines=True)

In [ ]:
# load file accuracy per bucket
df = pd.read_json('500samples_ACCURACY_aya23.json', lines=True)


# Function to calculate the standard error
def standard_error(group):
    return group.std() / np.sqrt(group.count())

# calculate average, max and min accuracy for each bucket and template 
df_agg2 = df_poem.groupby(['template_id', 'bucket']).agg(
    avg_accuracy=('accuracy','mean'),
    max_accuracy=('accuracy', 'max'),
    min_accuracy=('accuracy', 'min'),
    standard_error=('accuracy', standard_error)
).reset_index()

df_agg2

In [ ]:
# separate data for each template
template_1 = df[df['template'] == 2] #for leo-model use 1 and for aya 2
template_2 = df[df['template'] == 9] #for leo-model use 2 and for aya 9
template_3 = df[df['template'] == 10] #for leo-model use 3 and for aya 10

template_1agg = df_agg2[df_agg2['template_id'] == 2] #for leo-model use 1 and for aya 2
print(template_1agg)
template_2agg = df_agg2[df_agg2['template_id'] == 9] #for leo-model use 2 and for aya 9
template_3agg = df_agg2[df_agg2['template_id'] == 10] #for leo-model use 3 and for aya 10

# create three separate diagrams 
fig, axs = plt.subplots(3, 1, figsize=(10, 12))

# 1. line diagram for template 3

axs[0].errorbar(template_1['bucket'], 
             template_1['average_accuracy'], 
             yerr=template_1agg['standard_error'],  
             marker='o',
             markersize=3,
             color='b',
             ecolor='black',
             label='Template 3', 
             capsize=5)
axs[0].set_ylim(0, 60) #100 for leomistral
axs[0].set_title('Template 3 Accuracy')
axs[0].set_xlabel('Bucket')
axs[0].set_ylabel('Average Accuracy')

# 2. line diagram for template 9

axs[1].errorbar(template_2['bucket'], 
             template_2['average_accuracy'], 
             yerr=template_2agg['standard_error'],
             marker='o',
             markersize=3,
             color='g',
             ecolor='black',
             label='Template 9', 
             capsize=5)
axs[1].set_ylim(0, 60) #100 for leomistral
axs[1].set_title('Template 9 Accuracy')
axs[1].set_xlabel('Bucket')
axs[1].set_ylabel('Average Accuracy')

# 3. line diagram for template 10

axs[2].errorbar(template_3['bucket'], 
             template_3['average_accuracy'], 
             yerr=template_3agg['standard_error'], 
             marker='o',
             markersize=3,
             color='r', 
             ecolor='black',
             label='Template 10', 
             capsize=5)
axs[2].set_ylim(0, 60) #100 for leomistral
axs[2].set_title('Template 10 Accuracy')
axs[2].set_xlabel('Bucket')
axs[2].set_ylabel('Average Accuracy')

# adjust layout and show diagrams
plt.tight_layout()

# store image
plt.savefig('500samples_accuracy_standardError_AYA.png', dpi=300)
plt.show()



In [ ]:
#plot accuracy per bucket as line plot

# separate data for each template
template_1 = df[(df['template'] == 2) ] #for leo-model use 1 and for aya 2
template_2 = df[(df['template'] == 9) ] #for leo-model use 2 and for aya 9
template_3 = df[(df['template'] == 10) ] #for leo-model use 3 and for aya 10

template_1agg = df_agg2[df_agg2['template_id'] == 2] #for leo-model use 1 and for aya 2
template_2agg = df_agg2[df_agg2['template_id'] == 9] #for leo-model use 2 and for aya 9
template_3agg = df_agg2[df_agg2['template_id'] == 10] #for leo-model use 3 and for aya 10

print(template_1)
# create one plot
plt.figure(figsize=(10, 6))

# line for Template 3
plt.plot(template_1['bucket'], template_1['average_accuracy'], marker='o', color='b', label='Template 3')

# line for Template 9
plt.plot(template_2['bucket'], template_2['average_accuracy'], marker='o', color='g', label='Template 9')

# line for Template 10
plt.plot(template_3['bucket'], template_3['average_accuracy'], marker='o', color='r', label='Template 10')

# title and name for axes
plt.title('Accuracy per Bucket for All Templates')
plt.xlabel('Bucket')
plt.ylabel('Average Accuracy')

# add legend
plt.legend()

# adjust layout
plt.tight_layout()

# store image and show plot
plt.savefig('500samples_accuracy_AYA.png', dpi=300)
plt.show()

In [ ]:
#correlation plot

# Filter data for each template
template_1 = df_poem[(df_poem['template_id'] == 2)] #template_id either 2 (aya) or 1 (leo), data for template 3 
template_2 = df_poem[(df_poem['template_id'] == 9)] #template_id either 9 (aya) or 2(leo), data for template 9
template_3 = df_poem[(df_poem['template_id'] == 10)] #template_id either 10(aya) or 3(leo), data for template 10

# Define the y-axis range (accuracy)
min_accuracy = df_poem['accuracy'].min()
max_accuracy = df_poem['accuracy'].max()

# Create separate scatter plots for each template
plt.figure(figsize=(14, 12))

# Plot for Template 3
plt.subplot(3, 1, 1)
sns.scatterplot(x=template_1['year'], y=template_1['accuracy'], color='blue',alpha=0.1)
sns.regplot(x=template_1['year'], y=template_1['accuracy'], scatter=False, color='red', line_kws={"linestyle":"--"})
slope_1, intercept_1, r_value, p_value, std_err = stats.linregress(template_1['year'], template_1['accuracy'])
print(slope_1)
plt.title(f'Template 3: Years vs Accuracy (Slope: {slope_1:.2f})')
plt.ylim(0, 100)
plt.xlabel('Year')
plt.ylabel('Accuracy (%)')

# Plot for Template 9
plt.subplot(3, 1, 2)
sns.scatterplot(x=template_2['year'], y=template_2['accuracy'], color='blue',alpha=0.1)
sns.regplot(x=template_2['year'], y=template_2['accuracy'], scatter=False, color='red', line_kws={"linestyle":"--"})
slope_2, intercept_2, r_value, p_value, std_err = stats.linregress(template_2['year'], template_2['accuracy'])
plt.ylim(0, 100)
plt.title(f'Template 9: Years vs Accuracy (Slope: {slope_2:.2f})')
plt.xlabel('Year')
plt.ylabel('Accuracy (%)')

# Plot for Template 10
plt.subplot(3, 1, 3)
sns.scatterplot(x=template_3['year'], y=template_3['accuracy'], color='blue', alpha=0.1)
sns.regplot(x=template_3['year'], y=template_3['accuracy'], scatter=False, color='red', line_kws={"linestyle":"--"})
slope_3, intercept_3, r_value, p_value, std_err = stats.linregress(template_3['year'], template_3['accuracy'])
plt.ylim(0, 100)
plt.title(f'Template 10: Years vs Accuracy (Slope: {slope_3:.2f})')
plt.xlabel('Year')
plt.ylabel('Accuracy (%)')

# Show the plots
plt.tight_layout()

# store the image
plt.savefig('500samples_years_vs_accuracy_withSlope_AYA.png', dpi=300)
plt.show()


In [ ]:
#calculate accuracy per token per year
import json
df_results = pd.read_json('500samples_RESULTS_aya23_extracted.json', lines=True)

# Step 1: Calculate correct tags and total tokens per year and template
results = {}
count = 0

for _, row in df_results.iterrows():
    labels = row["labels"]
    extracted = row["extracted_pos_tags"]
    year = row["year"]
    template_id = row["template_id"]
    
    total = len(labels)  # Total tokens based on the number of labels (as requested)

    # Compare only if the lengths match, otherwise use the labels count for total
    correct = 0
    if len(labels) == len(extracted):
        correct = sum(1 for label, gen in zip(labels, extracted) if label == gen)
    else:
        #print(f"Length mismatch in year {year}, template {template_id}. Using labels for total token count.")
        count = count +1

    # Accumulate correct and total counts per (year, template_id)
    key = (year, template_id)
    
    if key not in results:
        results[key] = {"correct": 0, "total": 0}
    
    results[key]["correct"] += correct
    results[key]["total"] += total

# Step 2: Calculate accuracy per year and template
accuracy_per_year_template = {
    (year, template_id): (data["correct"] / data["total"]) * 100
    for (year, template_id), data in results.items()
}

# Save the accuracy data to a JSON file
# Convert tuple keys into separate fields in a list of dictionaries
json_ready_data = [
    {"year": key[0], "template": key[1], "accuracy": value} for key, value in accuracy_per_year_template.items()
]

# Save to JSON file
with open("accuracy_per_token_ayamodel.json", "w") as json_file:
    json.dump(json_ready_data, json_file, indent=4)

In [ ]:
#plot for accuracy per token

import matplotlib.pyplot as plt
from collections import defaultdict
import json
#plot for accuracy per token
#df = pd.read_json('accuracy_per_token_leomistral.json', lines=True)

with open('accuracy_per_token_ayamodel.json', 'r') as file:
    data = json.load(file)
df = pd.DataFrame(data)
# Convert 'template' column from string to integer
df['template'] = df['template'].astype(int)

# If you also want to convert 'year', you can do the same
df['year'] = df['year'].astype(int)
# Sort the DataFrame by 'year' in place
df.sort_values(by='year', inplace=True)

template_1 = df[df['template'] == 2] #1 if leomistral else 2, data for template 3
template_2 = df[df['template'] == 9] #2 if leomistral else 9, data for template 9
template_3 = df[df['template'] == 10] #3 if leomistral else 10, data for template 10

print(template_1)
# create three separate plots
fig, axs = plt.subplots(3, 1, figsize=(10, 12))

# 1. plot for Template 3
axs[0].plot(template_1['year'], template_1['accuracy'], marker='o', color='b')

#axs[0].set_ylim(0, 70)
axs[0].set_title('Template 3 Accuracy per token')
axs[0].set_xlabel('Year')
axs[0].set_ylabel('Accuracy per token')

# 2. plot for Template 9
axs[1].plot(template_2['year'], template_2['accuracy'], marker='o', color='g')

#axs[1].set_ylim(0, 70)
axs[1].set_title('Template 9 Accuracy per token')
axs[1].set_xlabel('Year')
axs[1].set_ylabel('Accuracy per token')

# 3. plot for Template 10
axs[2].plot(template_3['year'], template_3['accuracy'], marker='o', color='r')
#axs[2].set_ylim(0, 70)
axs[2].set_title('Template 10 Accuracy per token')
axs[2].set_xlabel('Year')
axs[2].set_ylabel('Accuracy per token')

# show plots
plt.tight_layout()

# and store image
plt.savefig('500samples_accuracy_aya.png', dpi=300)
plt.show()